In [ ]:
#Install dependencies
!pip install -q ultralytics opencv-python pyyaml tqdm boto3 mlflow scikit-learn

In [ ]:
import os
import yaml
import json
import mlflow
import time
import boto3
import glob
import traceback
import warnings
import logging
from botocore.client import Config
from urllib.parse import urlparse
import torch

import ultralytics
from ultralytics.data.utils import check_det_dataset
from ultralytics import YOLO
from ultralytics import settings
from ultralytics.utils import LOGGER, callbacks

# Create a custom filter for the specific warnings we want to suppress
class LabelWarningFilter(logging.Filter):
    def filter(self, record):
        msg = record.getMessage()
        # Only filter out messages about label classes exceeding dataset class count
        if "ignoring corrupt image/label: Label class" in msg and "exceeds dataset class count" in msg:
            return False  # Don't log these specific warnings
        return True  # Log everything else

settings.mlflow = False   # turn off Ultralytics’ autolog

# Apply the custom filter to the ultralytics logger
LOGGER.addFilter(LabelWarningFilter())

In [ ]:
# Set MinIO and MLflow configurations
MINIO_ENDPOINT_URL = "http://129.114.27.202:30000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "miniopassword"
MLFLOW_TRACKING_URI = "http://129.114.27.202:30938/"
BUCKET_NAME = "mlflow"
EXPERIMENT_NAME = "coco_runs"
MODEL_REGISTRY_NAME = "coco_detection_models"

# Configure MLflow to use MinIO
os.environ["MLFLOW_S3_ENDPOINT_URL"] = MINIO_ENDPOINT_URL
os.environ["AWS_ACCESS_KEY_ID"] = MINIO_ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = MINIO_SECRET_KEY
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI

# Configure MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Configure MinIO client
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT_URL,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Set up the experiment
try:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment:
        experiment_id = experiment.experiment_id
        print(f"Using existing MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")
    else:
        artifact_uri = f"s3://{BUCKET_NAME}/{EXPERIMENT_NAME}"
        experiment_id = mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=artifact_uri)
        print(f"Created new MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")
except Exception as e:
    print(f"Error setting up MLflow experiment: {e}")
    experiment_id = "0"

mlflow.set_experiment(EXPERIMENT_NAME)

Using existing MLflow experiment: coco_runs (ID: 1)


<Experiment: artifact_location='s3://mlflow/1', creation_time=1746919842723, experiment_id='1', last_update_time=1746919842723, lifecycle_stage='active', name='coco_runs', tags={}>

In [ ]:
# Download COCO via Ultralytics and get paths
coco_yaml = os.path.join(ultralytics.__path__[0], 'cfg', 'datasets', 'coco.yaml')
data_info = check_det_dataset(coco_yaml, autodownload=True)

print(f"COCO root: {data_info['path']}")
print(f" Train:   {data_info['train']}")
print(f" Val:     {data_info['val']}")

# Define traffic classes
TRAFFIC_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle',
    'bus', 'truck', 'traffic light', 'stop sign'
]

# Build traffic.yaml with string paths
traffic_spec = {
    'path':  str(data_info['path']),
    'train': str(data_info['train']),
    'val':   str(data_info['val']),
    'nc':    len(TRAFFIC_CLASSES),
    'names': TRAFFIC_CLASSES
}

with open('traffic.yaml', 'w') as f:
    yaml.safe_dump(traffic_spec, f, sort_keys=False)

print("▶ traffic.yaml created:")
print(yaml.safe_dump(traffic_spec, sort_keys=False))

COCO root: /content/datasets/coco
 Train:   /content/datasets/coco/train2017.txt
 Val:     /content/datasets/coco/val2017.txt
▶ traffic.yaml created:
path: /content/datasets/coco
train: /content/datasets/coco/train2017.txt
val: /content/datasets/coco/val2017.txt
nc: 8
names:
- person
- bicycle
- car
- motorcycle
- bus
- truck
- traffic light
- stop sign



In [ ]:
def generate_run_name():
    """Generate a unique run name with timestamp and model info."""
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    model_name = "yolov8n"
    return f"{model_name}_traffic_{timestamp}"

In [ ]:
def log_model_to_registry(run_id, model_path, model_name=MODEL_REGISTRY_NAME, stage="Development"):
    """Log the model to MLflow registry"""
    print(f"Registering model from {model_path}...")
    client = mlflow.tracking.MlflowClient()

    if not os.path.exists(model_path):
        print(f"❌ Model not found: {model_path}")
        return False

    try:
        # Create model in registry if it doesn't exist
        try:
            client.get_registered_model(model_name)
        except:
            client.create_registered_model(model_name)
            print(f"Created new model: {model_name}")

        # Log the model file as an artifact
        mlflow.log_artifact(model_path, "model")

        # Register the model
        model_uri = f"runs:/{run_id}/model/{os.path.basename(model_path)}"
        mv = mlflow.register_model(model_uri, model_name)

        # Transition stage
        client.transition_model_version_stage(model_name, mv.version, stage)
        print(f"✅ Registered model {model_name} version {mv.version} to {stage}")
        return True

    except Exception as e:
        print(f"❌ Model registration failed: {e}")
        traceback.print_exc()
        return False

In [ ]:
class MLflowCallback:
    def __init__(self):
        self.run_id = None
        self.start_time = None

    def __call__(self, trainer):
        pass

    def on_train_start(self, trainer):
        self.start_time = time.time()
        params = {
            "model_type": getattr(trainer.args, 'model', "yolov8n") if trainer else "yolov8n",
            "epochs":     getattr(trainer.args, 'epochs', 10) if trainer else 10,
            "batch_size": getattr(trainer.args, 'batch', 16) if trainer else 16,
            "image_size": getattr(trainer.args, 'imgsz', 640) if trainer else 640,
            "optimizer":  getattr(trainer.args, 'optimizer', "SGD") if trainer else "SGD",
            "learning_rate": getattr(trainer.args, 'lr0', 0.01) if trainer else 0.01,
            "classes":        TRAFFIC_CLASSES,
            "num_classes":    len(TRAFFIC_CLASSES),
            "dataset":        "COCO-traffic-subset",
            "pytorch_version": torch.__version__,
            "ultralytics_version": ultralytics.__version__,
        }
        mlflow.log_params(params)
        mlflow.log_artifact("traffic.yaml", "config")
        mlflow.set_tag("mlflow.source.name", "coco_train.py")
        self.run_id = mlflow.active_run().info.run_id
        print(f"✅ MLflow run started: {self.run_id}")

    def on_train_epoch_end(self, trainer):
        """Log metrics after each epoch"""
        current_epoch = trainer.epoch

        # Check if validation was run without relying on val_interval
        if hasattr(trainer, 'validator') and trainer.validator is not None:
            metrics = trainer.validator.metrics

            # Extract metrics from box
            if hasattr(metrics, 'box'):
                try:
                    # Map the metric attributes to the names you want to log
                    epoch_metrics = {
                        "metrics/precision": metrics.box.mp,
                        "metrics/recall": metrics.box.mr,
                        "metrics/mAP50": metrics.box.map50,
                        "metrics/mAP50-95": metrics.box.map
                    }

                    # Log to MLflow with the current epoch as the step
                    mlflow.log_metrics(epoch_metrics, step=current_epoch)

                    print(f"✅ Epoch {current_epoch} metrics logged to MLflow:")
                    for name, value in epoch_metrics.items():
                        print(f"   - {name}: {value:.4f}")
                except AttributeError as e:
                    print(f"⚠️ Error accessing box metrics: {e}")
                    # Debug: print all available attributes
                    print(f"Available box metrics attributes: {dir(metrics.box)}")
            else:
                print(f"⚠️ Metrics object doesn't have 'box' attribute at epoch {current_epoch}")

        # Always log training metrics
        try:
            # Get training loss metrics
            train_metrics = {
                "train/box_loss": float(trainer.loss_items[0]),
                "train/cls_loss": float(trainer.loss_items[1]),
                "train/dfl_loss": float(trainer.loss_items[2]),
                "train/total_loss": float(trainer.loss.item())
            }

            # Log training metrics to MLflow
            mlflow.log_metrics(train_metrics, step=current_epoch)

            print(f"✅ Epoch {current_epoch} training metrics logged to MLflow")
        except (IndexError, AttributeError) as e:
            print(f"⚠️ Error accessing training metrics: {e}")

    def on_train_end(self, trainer):
        """Log final metrics and artifacts"""
        duration = time.time() - self.start_time
        mlflow.log_metric("training_time_seconds", duration)

        # Log best model as artifact
        if hasattr(trainer, 'best') and trainer.best is not None:
            best_model_path = trainer.best
            if os.path.exists(best_model_path):
                mlflow.log_artifact(best_model_path, "models")
                print(f"✅ Best model logged as artifact: {best_model_path}")

        # Log PR curve if available
        pr_curve_path = os.path.join(trainer.save_dir, "PR_curve.png")
        if os.path.exists(pr_curve_path):
            mlflow.log_artifact(pr_curve_path, "evaluation")
            print(f"✅ PR curve logged as artifact")

        print("✅ Training ended, metrics logged")

In [ ]:
def main():
    # End any existing MLflow runs
    if mlflow.active_run():
        mlflow.end_run()

    # Start a new MLflow run
    with mlflow.start_run(run_name="yolov8_traffic_training") as run:
        run_id = run.info.run_id
        print(f"Started MLflow run: {run_id}")
        mlflow.set_tag("mlflow.source.name", "coco_train.py")
        # Get GPU information
        gpu_info = {
            "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
            "cuda_available": torch.cuda.is_available(),
        }

        # Add info for each GPU if available
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                gpu_info[f"gpu_{i}_name"] = torch.cuda.get_device_name(i)
                gpu_info[f"gpu_{i}_memory_total"] = torch.cuda.get_device_properties(i).total_memory / (1024**3)  # GB

            # Current device info
            current_device = torch.cuda.current_device()
            gpu_info["current_device_index"] = current_device
            gpu_info["current_device_name"] = torch.cuda.get_device_name(current_device)

        # Log system information and configuration
        mlflow.log_params({
            "model":          "yolov8n",
            "dataset":        "COCO-traffic-subset",
            "classes":        TRAFFIC_CLASSES,
            "epochs":         50,
            "image_size":     640,
            "batch_size":     16,
            "hardware":       gpu_info,
            "torch_version":  torch.__version__,
            **gpu_info 
        })

        # Create callback
        mlflow_callback = MLflowCallback()
        mlflow.log_param("training_start_time", time.strftime("%Y-%m-%d %H:%M:%S"))

        # Disable Ultralytics' built-in autolog
        from ultralytics import settings
        settings.mlflow = False

        # Initialize YOLO model
        print("Initializing YOLOv8 model...")
        model = YOLO('yolov8n.pt')

        # Register the callbacks
        for hook in ("on_train_start", "on_train_epoch_end", "on_train_end"):
            callback_method = getattr(mlflow_callback, hook)
            model.callbacks[hook].append(callback_method)

        try:
            # Start model training
            print("Starting model training...")
            results = model.train(
                data='traffic.yaml',
                epochs=50,
                imgsz=640,
                batch=16,
                project='runs/traffic',
                name='yolov8n_traffic_mlflow',
                val=True,  # Make sure validation runs after each epoch
                plots=True  # Generate plots for logging
            )

            # Log final metrics
            best_model_path = "runs/traffic/yolov8n_traffic_mlflow/weights/best.pt"

            # Use the results_dict to get final metrics
            if hasattr(results, 'results_dict'):
                final_metrics = {}
                # Map the results_dict keys to MLflow metric names
                metric_mapping = {
                    "metrics/precision(B)": "final/precision",
                    "metrics/recall(B)": "final/recall",
                    "metrics/mAP50(B)": "final/mAP50",
                    "metrics/mAP50-95(B)": "final/mAP50-95"
                }

                # Get metrics from results_dict with fallback to 0
                for result_key, mlflow_key in metric_mapping.items():
                    final_metrics[mlflow_key] = results.results_dict.get(result_key, 0)

                mlflow.log_metrics(final_metrics)
                print("Logged final metrics to MLflow")

            # Model registration
            if os.path.exists(best_model_path):
                log_model_to_registry(run_id, best_model_path)
            else:
                print(f"❌ No model found to register at {best_model_path}")

            # Print out links to MLflow UI
            artifact_url = f"{MLFLOW_TRACKING_URI}/#/experiments/{experiment_id}/runs/{run_id}/artifacts"
            print("\n✅ Training completed successfully!")
            print(f"   - Run ID:           {run_id}")
            print(f"   - View run:         {MLFLOW_TRACKING_URI}/#/experiments/{experiment_id}/runs/{run_id}")
            print(f"   - View artifacts:   {artifact_url}")
            print(f"   - View models:      {MLFLOW_TRACKING_URI}/#/models/{MODEL_REGISTRY_NAME}")

        except Exception as e:
            print(f"❌ Training failed: {e}")
            mlflow.log_param("error", str(e))
            mlflow.log_param("error_type", type(e).__name__)
            mlflow.set_tag("run_status", "failed")
            mlflow.set_tag("error_message", str(e)[:100])
            traceback.print_exc()

In [ ]:
# For testing and debugging
def log_model_structure():
    # Initialize a YOLOv8 model and debug its callback structure
    model = YOLO('yolov8n.pt')
    print(f"Model callbacks available hooks: {model.callbacks.keys()}")
    print(f"Trainer structure sample: {dir(model.trainer)}")

In [ ]:
main()

Started MLflow run: ccd4f18c17fa4979973f1de130bdd83a
Initializing YOLOv8 model...
Starting model training...
Ultralytics 8.3.131 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=traffic.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_traffic_mlflow19, nbs=64, nms=False, op

train: Scanning /content/datasets/coco/labels/train2017.cache... 117266 images, 1021 backgrounds, 107570 corrupt: 100%|██████████| 118287/118287 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 613.8±354.8 MB/s, size: 135.2 KB)


val: Scanning /content/datasets/coco/labels/val2017.cache... 4952 images, 48 backgrounds, 4545 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]


...

✅ Epoch 49 metrics logged to MLflow:
   - metrics/precision: 0.7948
   - metrics/recall: 0.6024
   - metrics/mAP50: 0.6974
   - metrics/mAP50-95: 0.4926
✅ Epoch 49 training metrics logged to MLflow


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:01<00:00,  8.15it/s]


                   all        455       1935      0.782       0.61      0.698      0.492

50 epochs completed in 0.929 hours.
Optimizer stripped from runs/traffic/yolov8n_traffic_mlflow19/weights/last.pt, 6.2MB
Optimizer stripped from runs/traffic/yolov8n_traffic_mlflow19/weights/best.pt, 6.2MB

Validating runs/traffic/yolov8n_traffic_mlflow19/weights/best.pt...
Ultralytics 8.3.131 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:03<00:00,  4.03it/s]


                   all        455       1935      0.764      0.605      0.697      0.493
                person        201        773      0.717      0.468      0.576      0.336
               bicycle         24         60      0.632        0.4      0.521      0.304
                   car        108        355      0.655      0.521      0.562      0.389
            motorcycle         81        195      0.809      0.626      0.741      0.486
                   bus         84        123      0.906      0.829      0.919       0.73
                 truck         91        143      0.853       0.72      0.821      0.683
         traffic light         88        112      0.897      0.795      0.863      0.651
             stop sign         86        174       0.64      0.481      0.574      0.366
Speed: 0.1ms preprocess, 0.4ms inference, 0.0ms loss, 1.2ms postprocess per image
Saving runs/traffic/yolov8n_traffic_mlflow19/predictions.json...

Evaluating pycocotools mAP using runs/traffic/yolov

## ______________________________________